# Recipe 02 — Image Encoding
> Encode image data into text descriptions using pluggable describe functions.

| | |
|---|---|
| **Module** | `anchor.multimodal.encoders` |
| **Key classes** | `ImageDescriptionEncoder`, `CompositeEncoder`, `TextEncoder` |
| **Difficulty** | Beginner |

In [ ]:
# Setup
from anchor.multimodal.encoders import (
    ImageDescriptionEncoder,
    CompositeEncoder,
    TextEncoder,
)
from anchor.multimodal import MultiModalContent, ModalityType

## Walkthrough

In [ ]:
# 1 — Define a mock image description function
# In production, this would call a vision model (e.g., Claude Vision)
def mock_describe(image_bytes: bytes) -> str:
    """Simulate an image description service."""
    size = len(image_bytes)
    return f"A photograph showing a sunset over mountains ({size} bytes analysed)"

print("mock_describe ready")
print(f"Test: {mock_describe(b'test')}")

In [ ]:
# 2 — Create the ImageDescriptionEncoder
# ImageDescriptionEncoder takes a callable that maps bytes -> str
img_encoder = ImageDescriptionEncoder(describe_fn=mock_describe)
print(f"Encoder: {type(img_encoder).__name__}")

In [ ]:
# Encode image content
img_content = MultiModalContent(
    modality=ModalityType.IMAGE,
    content=b"fake-image-bytes-png-data",
)

encoded = img_encoder.encode(img_content)
print(f"Input modality : {img_content.modality}")
print(f"Input size     : {len(img_content.content)} bytes")
print(f"Encoded output : {encoded}")

In [ ]:
# 3 — Build a CompositeEncoder with text and image support
# CompositeEncoder routes each modality to the right encoder
composite = CompositeEncoder(
    encoders={
        ModalityType.TEXT: TextEncoder(),
        ModalityType.IMAGE: img_encoder,
    },
)

print(f"Supported modalities: {list(composite.encoders.keys())}")

In [ ]:
# Encode text through the composite
text_content = MultiModalContent(
    modality=ModalityType.TEXT,
    content="A technical diagram of a neural network",
)
text_encoded = composite.encode(text_content)
print(f"Text encoded: {text_encoded}")

In [ ]:
# Encode image through the composite
image_encoded = composite.encode(img_content)
print(f"Image encoded: {image_encoded}")

In [ ]:
# Show that composite correctly routes by modality
print(f"Text via composite  : {text_encoded[:50]}")
print(f"Image via composite : {image_encoded[:50]}")
print(f"Same encoder?       : {text_encoded == image_encoded}")

In [ ]:
# 4 — Multiple images with different content
# Simulate encoding several images
image_payloads = [
    b"chart-revenue-q1-data",
    b"photo-team-offsite-jpg",
    b"screenshot-dashboard-png",
]

for i, payload in enumerate(image_payloads, 1):
    content = MultiModalContent(modality=ModalityType.IMAGE, content=payload)
    result = composite.encode(content)
    print(f"Image {i}: {result}")

In [ ]:
# Verify encoder isolation — text encoder ignores image data
text_result = composite.encode(text_content)
image_result = composite.encode(img_content)

print(f"Text result type  : {type(text_result).__name__}")
print(f"Image result type : {type(image_result).__name__}")
print(f"Results differ    : {text_result != image_result}")

## Key Takeaways
- `ImageDescriptionEncoder` converts image bytes to text via a `describe_fn` callable.
- Swap `mock_describe` for a real vision API (e.g., Claude Vision) in production.
- `CompositeEncoder` dispatches to the correct encoder based on `ModalityType`.
- Text and image modalities can be mixed freely within the same pipeline.

**Next:** [Table Extraction](03_table_extraction.ipynb)